# 02 — Exploring a GNN Model

In notebook 01 we scanned a repository and built a `ProgramGraph`. This notebook takes the next step: run the translation + state-space + export stages so COGANT emits a full **GNN markdown** file, then parse that file with `cogant.reverse.parse_gnn` and visualise the A/B/C/D matrices.

The GNN (Generalized Notation Notation) markdown is the canonical interchange format for Active Inference models — every section maps to a concept from the Active Inference Institute's spec.

**Run from the `cogant/` directory:**
```bash
uv run jupyter nbconvert --to notebook --execute docs/notebooks/02_explore_gnn.ipynb
```

## 1. Setup — run the forward pipeline and export

We reuse the calculator fixture from notebook 01. `session.export_all(...)` writes a full GNN bundle to a temp workspace, including `model.gnn.md`.

In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

from cogant.api.session import Session

CANDIDATE_FIXTURES = [
    Path("tests/fixtures/calculator/"),
    Path("examples/control_positive/calculator/"),
]
FIXTURE: Path | None = next((p.resolve() for p in CANDIDATE_FIXTURES if p.exists()), None)
if FIXTURE is None:
    raise FileNotFoundError("No calculator fixture found; run from the cogant/ directory.")
print(f"Fixture: {FIXTURE}")

In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="cogant_nb02_"))
print(f"Workdir: {workdir}")

session = Session(repo_path=FIXTURE)
session.export_all(str(workdir))

print("Exported artifacts:")
for key, path in sorted(session.export_artifacts.items()):
    print(f"  {key:<20} {path}")

## 2. Locate the GNN markdown file

`export_all` writes `model.gnn.md` under `<workdir>/gnn_package/`. We look for it and read the raw text so you can see what a real GNN document looks like.

In [ ]:
gnn_candidates = sorted(workdir.rglob("model.gnn.md"))
if not gnn_candidates:
    raise FileNotFoundError(f"No model.gnn.md under {workdir}. Export keys: {list(session.export_artifacts)}")

gnn_path = gnn_candidates[0]
gnn_text = gnn_path.read_text(encoding="utf-8")
print(f"Loaded: {gnn_path} ({len(gnn_text)} chars)")
print("First 30 lines:")
print("\n".join(gnn_text.splitlines()[:30]))

## 3. Parse the GNN file

`cogant.reverse.parse_gnn` is the canonical parser — the same one used by the reverse synthesis pipeline in notebook 03. It returns a `ReverseGNNModel` with structured access to every section.

In [ ]:
from cogant.reverse import parse_gnn

model = parse_gnn(gnn_path)

print(f"Model name     : {model.model_name}")
print(f"Hidden states  : {model.hidden_states}")
print(f"Observations   : {model.observations}")
print(f"Actions        : {model.actions}")
print(f"Policies       : {model.policies}")
print(f"Constraints    : {model.constraints}")
print(f"Cardinalities  : {model.cardinalities}")

### StateSpaceBlock and ActInfOntologyAnnotation

These two sections are the semantic backbone of a GNN file. `StateSpaceBlock` declares every variable with its cardinality and numeric type; `ActInfOntologyAnnotation` maps each variable to an Active Inference role (`HiddenState`, `LikelihoodMatrix`, `PriorBelief`, …).

In [ ]:
def extract_section(text: str, header: str) -> str:
    lines = text.splitlines()
    out: list[str] = []
    in_section = False
    for line in lines:
        stripped = line.strip()
        if stripped.startswith("## "):
            if in_section:
                break
            if stripped[3:].strip() == header:
                in_section = True
            continue
        if in_section:
            out.append(line)
    return "\n".join(out).strip()

print("## StateSpaceBlock")
print(extract_section(gnn_text, "StateSpaceBlock") or "(empty)")
print()
print("## ActInfOntologyAnnotation")
print(extract_section(gnn_text, "ActInfOntologyAnnotation") or "(empty)")

## 4. Visualise the A/B/C/D matrices

The parser collects the four canonical Active Inference matrices into `model.A`, `model.B`, `model.C`, and `model.D`. Depending on the source repository some of these may be empty — e.g. a model with no observation modality has no `A`. We plot whichever ones exist as heatmaps.

In [ ]:
import numpy as np

matrices = {
    "A (likelihood P(o|s))": model.A,
    "B (transitions P(s'|s,a))": model.B,
    "C (preferences over obs)": model.C,
    "D (prior over states)": model.D,
}

shapes = {}
for name, arr in matrices.items():
    try:
        np_arr = np.asarray(arr, dtype=float) if arr is not None else np.empty((0,))
    except (TypeError, ValueError):
        np_arr = np.empty((0,))
    shapes[name] = np_arr.shape
    print(f"  {name:<30} shape={np_arr.shape}")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for ax, (name, arr) in zip(axes.flat, matrices.items()):
    try:
        np_arr = np.asarray(arr, dtype=float)
    except (TypeError, ValueError):
        np_arr = np.empty((0,))
    if np_arr.size == 0:
        ax.text(0.5, 0.5, "(empty)", ha="center", va="center")
        ax.set_axis_off()
    else:
        # Flatten any trailing singleton dims for plotting.
        plot_arr = np.atleast_2d(np.squeeze(np_arr))
        if plot_arr.ndim == 1:
            plot_arr = plot_arr[np.newaxis, :]
        if plot_arr.ndim > 2:
            plot_arr = plot_arr.reshape(plot_arr.shape[0], -1)
        im = ax.imshow(plot_arr, aspect="auto", cmap="viridis")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(name)

plt.tight_layout()
fig

## 5. Takeaways

- A GNN file is self-describing: the `StateSpaceBlock` and `ActInfOntologyAnnotation` sections alone are enough to interpret the rest.
- Empty matrices are **normal** — a simple calculator has no observation modality, so it has no `A` or `C`. Richer fixtures (e.g. `examples/control_positive/event_pipeline`) populate all four.
- `parse_gnn` is the single source of truth: if you can load your GNN with it, the reverse pipeline can synthesise Python from it (notebook 03).